In [1]:
!pip install transformers accelerate peft bitsandbytes "dspy-ai==3.3.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.2/290.2 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: gepa
    Found existing installation: gepa 0.1.1
    Uninstalling gepa-0.1.1:
      Successfully uninstalled gepa-0.1.1


In [2]:
!pip install -U "torchao>=0.17.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 63.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


# Loading 0.5B Model

In [3]:
import os, gc, json, re, time, random
import torch
import numpy as np
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import dspy
import copy

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

BASE_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
# NOTE: reversed pipeline -- KD now runs FIRST on a plain base, so there is no
# pre-existing QLoRA adapter to point at here (that only exists after notebook 2).

def flush_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

def gpu_report(tag=""):
    for i in range(torch.cuda.device_count()):
        used = torch.cuda.memory_allocated(i) / 1e9
        total = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"[{tag}] GPU {i}: {used:.2f} GB / {total:.2f} GB")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [4]:
bnb_config_t4 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4 has native fp16 tensor cores, no bf16 support
    bnb_4bit_use_double_quant=True,
)

# Loading 7B model

In [5]:
class TrainingFormatAdapter(dspy.ChatAdapter):
    def format(self, signature, demos, inputs):
        output_fields = list(signature.output_fields.keys())
        if output_fields != ["result"]:
            return super().format(signature, demos, inputs)
        messages = [{"role": "system", "content": signature.instructions}]
        for demo in demos:
            messages.append({"role": "user", "content": f"Report:\n{demo['report']}"})
            messages.append({"role": "assistant", "content": demo["result"]})
        messages.append({"role": "user", "content": f"Report:\n{inputs['report']}"})
        return messages

    def parse(self, signature, completion):
        output_fields = list(signature.output_fields.keys())
        if output_fields == ["result"]:
            return {"result": completion}
        return super().parse(signature, completion)

class LocalHFLM(dspy.LM):
    def __init__(self, model, tokenizer, model_name="local-hf", max_new_tokens=512, temperature=0.0):
        super().__init__(model=model_name)
        self.hf_model = model
        self.tokenizer = tokenizer
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature

    def forward(self, prompt=None, messages=None, **kwargs):
        if messages is None:
            messages = [{"role": "user", "content": prompt}]
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(text, return_tensors="pt").to(self.hf_model.device)
        do_sample = self.temperature > 0
        with torch.no_grad():
            output_ids = self.hf_model.generate(
                **inputs, max_new_tokens=self.max_new_tokens, do_sample=do_sample,
                temperature=self.temperature if do_sample else None,
                pad_token_id=self.tokenizer.pad_token_id, eos_token_id=self.tokenizer.eos_token_id,
            )
        generated = output_ids[0][inputs["input_ids"].shape[-1]:]
        completion = self.tokenizer.decode(generated, skip_special_tokens=True).strip()
        from litellm.types.utils import ModelResponse, Choices, Message
        return ModelResponse(choices=[Choices(message=Message(role="assistant", content=completion))], model=self.model)

    def __deepcopy__(self, memo):
        new_instance = LocalHFLM(
            model=self.hf_model, tokenizer=self.tokenizer, model_name=self.model,
            max_new_tokens=self.max_new_tokens, temperature=self.temperature,
        )
        new_instance.history = []
        new_instance.kwargs = copy.deepcopy(self.kwargs, memo) if hasattr(self, "kwargs") else {}
        return new_instance

In [6]:
PROPOSER_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

proposer_tok = AutoTokenizer.from_pretrained(PROPOSER_MODEL_ID, trust_remote_code=True)
proposer_tok.pad_token = proposer_tok.eos_token
proposer_tok.padding_side = "left"  # required for batched generation

proposer_model = AutoModelForCausalLM.from_pretrained(
    PROPOSER_MODEL_ID, quantization_config=bnb_config_t4, device_map={"": 0},
    trust_remote_code=True, attn_implementation="sdpa",
)
proposer_model.eval()
gpu_report("after proposer reload (4-bit, fp16 compute, sdpa)")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[after proposer reload (4-bit, fp16 compute, sdpa)] GPU 0: 5.58 GB / 15.64 GB
[after proposer reload (4-bit, fp16 compute, sdpa)] GPU 1: 0.00 GB / 15.64 GB


# Batched Generation

In [7]:
def build_batch_inputs(tokenizer, reports, acr_index):
    """Tokenize a batch of reports with left-padding; track individual prompt lengths."""
    all_messages = [build_chat_messages(r["free_text"], acr_index) for r in reports]
    texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in all_messages]
    encoded = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False)
    return encoded

def generate_batch_with_topk_logprobs(model, tokenizer, reports, acr_index, k=20, max_new_tokens=768):
    encoded = build_batch_inputs(tokenizer, reports, acr_index)
    encoded = {kk: v.to(model.device) for kk, v in encoded.items()}
    prompt_len = encoded["input_ids"].shape[1]  # same for all, due to left-padding

    with torch.no_grad():
        out = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            output_scores=True,
            return_dict_in_generate=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    batch_size = len(reports)
    generated_ids_batch = out.sequences[:, prompt_len:]  # (B, gen_len)

    results = []
    for b in range(batch_size):
        gen_ids = generated_ids_batch[b]
        eos_positions = (gen_ids == tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
        stop_idx = eos_positions[0].item() + 1 if len(eos_positions) > 0 else len(gen_ids)
        gen_ids_trimmed = gen_ids[:stop_idx]

        generated_text = tokenizer.decode(gen_ids_trimmed, skip_special_tokens=True).strip()

        token_distributions = []
        for step_idx in range(stop_idx):
            step_logits = out.scores[step_idx][b]  # (V,)
            probs = F.softmax(step_logits.float(), dim=-1)
            topk_probs, topk_ids = torch.topk(probs, k)
            token_distributions.append({
                "position": step_idx,
                "generated_token_id": gen_ids_trimmed[step_idx].item(),
                "generated_token_str": tokenizer.decode([gen_ids_trimmed[step_idx].item()]),
                "topk_token_ids": topk_ids.tolist(),
                "topk_probs": [round(p, 6) for p in topk_probs.tolist()],
                "topk_mass": round(topk_probs.sum().item(), 6),
            })
        results.append((generated_text, token_distributions))

    return results

def load_processed_ids(path):
    processed = set()
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                try:
                    processed.add(json.loads(line)["report_id"])
                except Exception:
                    continue
    return processed

# Create train and test pool

In [8]:
import json

# --- Load your pre-split abdomen train/test JSONL files ---
ABDOMEN_TRAIN_PATH = "/kaggle/input/datasets/mythreyee1006/train-dataset-abdomen-new/train_records_abdomenCT.jsonl"  # adjust path
ABDOMEN_TEST_PATH = "/kaggle/input/datasets/mythreyee1006/test-dataset-abdomen-final/test_records_abdomenCT.jsonl"    # adjust path



def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records


train_records_raw = load_jsonl(ABDOMEN_TRAIN_PATH)
test_records_raw = load_jsonl(ABDOMEN_TEST_PATH)
print(f"Train records loaded: {len(train_records_raw)}")
print(f"Test records loaded: {len(test_records_raw)}")


# --- Reconstruct {report_id, free_text, gold} from the messages format ---
def extract_structured(record, idx, prefix):
    messages = record["messages"]
    user_msg = next(m["content"] for m in messages if m["role"] == "user")
    assistant_msg = next(m["content"] for m in messages if m["role"] == "assistant")
    free_text = user_msg.split("Report:\n", 1)[-1]  # strip the "Report:\n" prefix
    gold = json.loads(assistant_msg)  # {"contains_IF": bool, "incidental_sentences": [...]}
    return {
        "report_id": f"{prefix}{idx:04d}",  # synthetic ID, prefixed by split so train/test IDs can't collide
        "free_text": free_text,
        "gold": gold,
    }


train_structured_raw = [extract_structured(r, i, "TRAIN") for i, r in enumerate(train_records_raw)]
test_dataset = [extract_structured(r, i, "TEST") for i, r in enumerate(test_records_raw)]
print(f"Structured train records: {len(train_structured_raw)}")
print(f"Structured test records: {len(test_dataset)}")

# --- Dedup/overlap check ---
# Since IDs are prefixed by split, report_id collisions between train/test are
# impossible by construction -- this check instead catches duplicate FREE TEXT
# across the two files (e.g. the same report accidentally present in both),
# which a report_id-only check would miss entirely here.
test_texts = {r["free_text"].strip() for r in test_dataset}
test_ids = {r["report_id"] for r in test_dataset}

before = len(train_structured_raw)
seen_ids = set()
seen_texts = set()
train_pool = []
dupes_within_train = 0
overlap_with_test = 0

for r in train_structured_raw:
    text_key = r["free_text"].strip()
    if r["report_id"] in test_ids or text_key in test_texts:
        overlap_with_test += 1
        continue
    if r["report_id"] in seen_ids or text_key in seen_texts:
        dupes_within_train += 1
        continue
    seen_ids.add(r["report_id"])
    seen_texts.add(text_key)
    train_pool.append(r)

print(f"Before: {before}")
print(f"Dropped (overlap with test_dataset, by ID or exact text): {overlap_with_test}")
print(f"Dropped (duplicate within train pool, by ID or exact text): {dupes_within_train}")
print(f"Clean train pool: {len(train_pool)}")

if overlap_with_test > 0:
    print("\n⚠️  Overlap found and removed — check whether train/test were meant to be disjoint.")

Train records loaded: 1184
Test records loaded: 100
Structured train records: 1184
Structured test records: 100
Before: 1184
Dropped (overlap with test_dataset, by ID or exact text): 0
Dropped (duplicate within train pool, by ID or exact text): 0
Clean train pool: 1184


In [9]:
pos = sum(r["gold"]["contains_IF"] for r in train_pool)
neg = sum(not r["gold"]["contains_IF"] for r in train_pool)

print(f"Positive: {pos}")
print(f"Negative: {neg}")
print(f"Negative:Positive = {neg}:{pos}")
print(f"Negative/Positive ratio = {neg / pos:.2f}")

Positive: 459
Negative: 725
Negative:Positive = 725:459
Negative/Positive ratio = 1.58


# Load ACR guidelines and condense into prompt-ready context

In [10]:
with open("/kaggle/input/datasets/mythreyeehari20/acr-rules-abdomen/Findings_Extracted_WhitePapers.json") as f:
    acr_guidelines = json.load(f)


def format_acr_context(guidelines):
    """Condense the ACR abdomen findings into a compact prompt-ready context."""
    
    blocks = []

    for finding in guidelines["findings"]:
        feat_str = "; ".join(finding["features"])

        blocks.append(
            f"- [{finding['finding_id']}] {finding['finding_name']}: "
            f"{finding['description']} "
            f"Key features: {feat_str}"
        )

    return "\n".join(blocks)


ACR_CONTEXT = format_acr_context(acr_guidelines)

print(
    f"ACR context length: {len(ACR_CONTEXT):,} chars "
    f"(~{len(ACR_CONTEXT)//3.5:,.0f} tokens)"
)

ACR context length: 16,438 chars (~4,696 tokens)


# Select few-shot exemplars from Train Pool

In [11]:
def select_fewshot_demos(pool, n_negative=1, n_single=2, n_multi=2, seed=42):
    random.seed(seed)

    negative = [
        r for r in pool
        if r["gold"]["contains_IF"] is False
    ]

    single = [
        r for r in pool
        if len(r["gold"].get("incidental_sentences", [])) == 1
    ]

    multi = [
        r for r in pool
        if len(r["gold"].get("incidental_sentences", [])) >= 2
    ]

    demos = (
        random.sample(negative, min(n_negative, len(negative))) +
        random.sample(single, min(n_single, len(single))) +
        random.sample(multi, min(n_multi, len(multi)))
    )

    random.shuffle(demos)
    return demos


fewshot_raw = select_fewshot_demos(train_pool)

for d in fewshot_raw:
    print(
        d["report_id"],
        "-> contains_IF:", d["gold"]["contains_IF"],
        "| n_sentences:", len(d["gold"].get("incidental_sentences", []))
    )

TRAIN0477 -> contains_IF: True | n_sentences: 2
TRAIN1064 -> contains_IF: False | n_sentences: 0
TRAIN0092 -> contains_IF: True | n_sentences: 1
TRAIN0421 -> contains_IF: True | n_sentences: 3
TRAIN0272 -> contains_IF: True | n_sentences: 1


# Build the signature with ACR context baked into instructions

In [12]:
# Template with a placeholder for the per-report filtered ACR context
TASK_INSTRUCTIONS_TEMPLATE = (
    "You are a clinical assistant specialized in abdominopelvic radiology. Your task is to identify "
    "INCIDENTAL findings in a free-text abdominopelvic CT report — findings unrelated to the report's "
    "primary clinical indication, per ACR Incidental Findings Committee guidelines.\n\n"
    "Use the reference guidelines below to judge whether a finding is clinically incidental "
    "(e.g. small stable nodules, benign-appearing lymph nodes, calcifications) versus a primary/"
    "expected finding tied to the report's main indication.\n\n"
    "Extract the EXACT sentence(s) from the report that describe incidental findings — do not "
    "paraphrase or summarize. If no incidental findings are present, return an empty list.\n\n"
    "REFERENCE GUIDELINES:\n{acr_context}\n\n"
    "OUTPUT FORMAT:\n"
    "Return ONLY one JSON object and nothing else.\n"
    "The JSON object MUST contain exactly these two fields: "
    "contains_IF and incidental_sentences.\n"
    "contains_IF MUST be a boolean: true or false.\n"
    "incidental_sentences MUST be a JSON array of STRINGS, not objects.\n"
    "Each string must be an EXACT sentence copied from the report.\n"
    "Do NOT add fields such as sentence, if, is_incidental, findings, "
    "reference_guidelines, or any other fields.\n"
    "If there are no incidental findings, use an empty array and set contains_IF to false.\n"
    "If incidental findings are present, set contains_IF to true and include only "
    "the exact sentences containing those findings.\n"
    "Required format:\n"
    "{{\"contains_IF\": false, \"incidental_sentences\": []}}\n"
)

def build_dynamic_program(filtered_context):
    instructions = TASK_INSTRUCTIONS_TEMPLATE.format(acr_context=filtered_context)

    class DynamicExtractFindings(dspy.Signature):
        __doc__ = instructions
        report = dspy.InputField()
        result = dspy.OutputField(desc='{"contains_IF": true/false, "incidental_sentences": []}')

    program = dspy.Predict(DynamicExtractFindings)
    program.demos = few_shot_demos
    return program

# Wire up the teacher LM and sanity-check on one report

In [13]:
import re


def condense_finding(finding):
    """One compact line per finding: name + terse features only, no narrative prose."""
    feat_str = "; ".join(finding["features"])
    return f"[{finding['finding_id']}] {finding['finding_name']} — {feat_str}"


def build_condensed_index(guidelines):
    """Build a searchable index from the abdomen ACR findings."""
    
    index = []

    for finding in guidelines["findings"]:
        searchable = " ".join([
            finding["finding_name"],
            " ".join(finding["features"]),
        ]).lower()

        index.append({
            "finding_id": finding["finding_id"],
            "searchable_text": searchable,
            "condensed_line": condense_finding(finding),
        })

    return index


ACR_INDEX = build_condensed_index(acr_guidelines)

print(f"Indexed {len(ACR_INDEX)} findings")


# Quick size check
full_condensed = "\n".join(
    f["condensed_line"] for f in ACR_INDEX
)

print(
    f"Full condensed size: {len(full_condensed):,} chars "
    f"(vs original ~{len(ACR_CONTEXT):,} chars)"
)


STOPWORDS = {
    "the", "a", "an", "of", "in", "on", "to", "and", "or",
    "with", "is", "are", "was", "were", "at", "for", "by",
    "as", "be", "no", "not", "also", "this", "that",
    "been", "has", "have"
}


def tokenize(text):
    words = re.findall(r"[a-z]+", text.lower())
    return set(
        w for w in words
        if w not in STOPWORDS and len(w) > 2
    )


def retrieve_relevant_findings(
    report_text,
    index,
    top_k=20,
    min_overlap=1
):
    """Keyword-overlap retrieval: score each finding by token overlap with report text."""

    report_tokens = tokenize(report_text)

    scored = []

    for entry in index:
        finding_tokens = tokenize(entry["searchable_text"])

        overlap = len(report_tokens & finding_tokens)

        if overlap >= min_overlap:
            scored.append((overlap, entry))

    scored.sort(key=lambda x: -x[0])

    top = [
        entry
        for _, entry in scored[:top_k]
    ]

    return top


def build_filtered_acr_context(
    report_text,
    index,
    top_k=20
):
    relevant = retrieve_relevant_findings(
        report_text,
        index,
        top_k=top_k
    )

    if not relevant:
        # Fallback: don't send an empty context
        return full_condensed

    return "\n".join(
        e["condensed_line"]
        for e in relevant
    )


# Test on one abdomen report
sample_report = train_pool[0]["free_text"]

filtered_context = build_filtered_acr_context(
    sample_report,
    ACR_INDEX,
    top_k=20
)

print(
    f"\nFiltered context size: "
    f"{len(filtered_context):,} chars"
)

print("\nRetrieved ACR findings:")
print(filtered_context)

Indexed 27 findings
Full condensed size: 13,332 chars (vs original ~16,438 chars)

Filtered context size: 13,332 chars

Retrieved ACR findings:
[AP-F001] Benign-appearing adnexal cyst — Oval or round shape [cite: 1932]; Unilocular mass of uniform fluid signal and attenuation [cite: 1932]; Regular or imperceptible wall [cite: 1932]; Absence of solid areas or mural nodules [cite: 1932]; Maximum diameter is strictly less than 10 cm [cite: 1932]; May contain layering hemorrhage if the patient is premenopausal [cite: 1933]
[AP-F002] Probably benign adnexal cyst — Displays angulated margins [cite: 1938]; Is not round or oval in shape [cite: 1938]; A portion of the cyst is poorly imaged, such as being obscured by metal streak artifact [cite: 1938]; Image demonstrates reduced signal-to-noise ratio, often due to technical parameters or lack of intravenous contrast [cite: 1938, 1939]
[AP-F003] Adnexal mass with suspicious features — Presence of a solid component [cite: 2005]; Presence of a mural

In [14]:
prompt_lm = LocalHFLM(proposer_model, proposer_tok, model_name="qwen7b-proposer",
                       max_new_tokens=768, temperature=0.0)

In [15]:
few_shot_demos = [
    dspy.Example(
        report=r["free_text"],
        result=json.dumps(r["gold"])
    ).with_inputs("report")
    for r in fewshot_raw
]

print(f"Few-shot demos created: {len(few_shot_demos)}")

Few-shot demos created: 5


In [16]:
dspy.configure(
    lm=prompt_lm,
    adapter=TrainingFormatAdapter()
)

sample = train_pool[0]

filtered_context = build_filtered_acr_context(
    sample["free_text"],
    ACR_INDEX,
    top_k=20
)

dynamic_program = build_dynamic_program(
    filtered_context
)

pred = dynamic_program(
    report=sample["free_text"]
)

print("PREDICTED:", pred.result)
print("GOLD:     ", json.dumps(sample["gold"]))

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


PREDICTED: {"contains_IF": false, "incidental_sentences": []}
GOLD:      {"contains_IF": false, "incidental_sentences": []}


In [17]:
def parse_output(text):
    if not text:
        return None
    matches = re.findall(r'\{[^{}]*\}', text, re.DOTALL)
    if not matches:
        return None
    for candidate in reversed(matches):
        try:
            parsed = json.loads(candidate)
            if "incidental_sentences" in parsed:
                return parsed
        except Exception:
            continue
    return None

# Prompt Builder

In [18]:
with open("/kaggle/input/datasets/mythreyeehari20/kd-dataset-abdomen/CT_ABD_REPORTS_UNANNOTATED.json") as f:
    data = json.load(f)

unannotated_reports = data["reports"]

print("Number of reports:", len(unannotated_reports))

def build_chat_messages(report_text, acr_index, top_k=20):
    filtered_context = build_filtered_acr_context(report_text, acr_index, top_k=top_k)
    instructions = TASK_INSTRUCTIONS_TEMPLATE.format(acr_context=filtered_context)

    messages = [{"role": "system", "content": instructions}]
    for demo in fewshot_raw:
        messages.append({"role": "user", "content": f"Report:\n{demo['free_text']}"})
        messages.append({"role": "assistant", "content": json.dumps(demo["gold"])})
    messages.append({"role": "user", "content": f"Report:\n{report_text}"})
    return messages

Number of reports: 1500


In [19]:
def build_batch_inputs(tokenizer, reports, acr_index, retrieval_top_k=10):
    all_messages = [build_chat_messages(r["free_text"], acr_index, top_k=retrieval_top_k) for r in reports]
    texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in all_messages]
    encoded = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False)
    return encoded

def generate_batch_with_topk_logprobs(model, tokenizer, reports, acr_index, k=20, max_new_tokens=768, retrieval_top_k=10):
    encoded = build_batch_inputs(tokenizer, reports, acr_index, retrieval_top_k=retrieval_top_k)
    encoded = {kk: v.to(model.device) for kk, v in encoded.items()}
    prompt_len = encoded["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            output_scores=True,
            return_dict_in_generate=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    batch_size = len(reports)
    generated_ids_batch = out.sequences[:, prompt_len:]

    results = []
    for b in range(batch_size):
        gen_ids = generated_ids_batch[b]
        eos_positions = (gen_ids == tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
        stop_idx = eos_positions[0].item() + 1 if len(eos_positions) > 0 else len(gen_ids)
        gen_ids_trimmed = gen_ids[:stop_idx]

        generated_text = tokenizer.decode(gen_ids_trimmed, skip_special_tokens=True).strip()

        token_distributions = []
        for step_idx in range(stop_idx):
            step_logits = out.scores[step_idx][b]
            probs = F.softmax(step_logits.float(), dim=-1)
            topk_probs, topk_ids = torch.topk(probs, k)
            token_distributions.append({
                "position": step_idx,
                "generated_token_id": gen_ids_trimmed[step_idx].item(),
                "generated_token_str": tokenizer.decode([gen_ids_trimmed[step_idx].item()]),
                "topk_token_ids": topk_ids.tolist(),
                "topk_probs": [round(p, 6) for p in topk_probs.tolist()],
                "topk_mass": round(topk_probs.sum().item(), 6),
            })
        results.append((generated_text, token_distributions))

    return results

# Generation + top-k capture function

In [20]:
import torch.nn.functional as F

def generate_with_topk_logprobs(model, tokenizer, messages, k=20, max_new_tokens=768):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # greedy — matches temperature=0.0 decision
            output_scores=True,
            return_dict_in_generate=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = out.sequences[0][input_len:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    token_distributions = []
    for step_idx, step_logits in enumerate(out.scores):
        probs = F.softmax(step_logits[0].float(), dim=-1)
        topk_probs, topk_ids = torch.topk(probs, k)
        token_distributions.append({
            "position": step_idx,
            "generated_token_id": generated_ids[step_idx].item(),
            "generated_token_str": tokenizer.decode([generated_ids[step_idx].item()]),
            "topk_token_ids": topk_ids.tolist(),
            "topk_probs": [round(p, 6) for p in topk_probs.tolist()],
            "topk_mass": round(topk_probs.sum().item(), 6),  # for renorm/other-bucket downstream
        })

    return generated_text, token_distributions

# Empirically pick k

In [21]:
def calibrate_k(model, tokenizer, sample_records, acr_index, k_max=50, k_candidates=(5, 10, 20, 30, 50)):
    mass_at_k = {k: [] for k in k_candidates}
    for rec in tqdm(sample_records, desc="k calibration"):
        messages = build_chat_messages(rec["free_text"], acr_index)
        _, dists = generate_with_topk_logprobs(model, tokenizer, messages, k=k_max)
        for step in dists:
            sorted_probs = step["topk_probs"]
            for k in k_candidates:
                mass_at_k[k].append(sum(sorted_probs[:k]))
    for k in k_candidates:
        avg_mass = sum(mass_at_k[k]) / len(mass_at_k[k])
        print(f"k={k}: avg cumulative top-k mass = {avg_mass:.4f}")
    return mass_at_k

calibration_sample = random.sample(unannotated_reports, 20)
_ = calibrate_k(proposer_model, proposer_tok, calibration_sample, ACR_INDEX)

k calibration: 100%|██████████| 20/20 [03:15<00:00,  9.80s/it]

k=5: avg cumulative top-k mass = 0.9984
k=10: avg cumulative top-k mass = 0.9989
k=20: avg cumulative top-k mass = 0.9991
k=30: avg cumulative top-k mass = 0.9992
k=50: avg cumulative top-k mass = 0.9993


# Full run over 1500, with checkpointing

In [22]:
import torch.nn.functional as F

BATCH_SIZE = 1  # start conservative given ~10GB free headroom; increase if stable
OUTPUT_PATH = "/kaggle/working/kd_annotated_dataset.jsonl"
CHOSEN_K = 10

processed_ids = load_processed_ids(OUTPUT_PATH)
print(f"Already processed: {len(processed_ids)} / {len(unannotated_reports)}")

remaining = [r for r in unannotated_reports if r["report_id"] not in processed_ids]

with open(OUTPUT_PATH, "a") as f_out:
    for i in tqdm(range(0, len(remaining), BATCH_SIZE), desc="Batched teacher annotation"):
        batch_reports = remaining[i:i + BATCH_SIZE]
        try:
            batch_results = generate_batch_with_topk_logprobs(
                proposer_model, proposer_tok, batch_reports, ACR_INDEX, k=CHOSEN_K, retrieval_top_k=10
            )
        except torch.cuda.OutOfMemoryError:
            print(f"OOM on batch starting at {i} — consider lowering BATCH_SIZE")
            flush_gpu()
            continue
        except Exception as e:
            print(f"Batch failed at {i}: {e}")
            continue

        for rec, (gen_text, token_dists) in zip(batch_reports, batch_results):
            parsed = parse_output(gen_text)
            record = {
                "report_id": rec["report_id"],
                "free_text": rec["free_text"],
                "annotated_output_raw": gen_text,
                "annotated_output_parsed": parsed,
                "parse_failed": parsed is None,
                "topk_k": CHOSEN_K,
                "token_distributions": token_dists,
            }
            f_out.write(json.dumps(record) + "\n")
        f_out.flush()

print("Done.")

Already processed: 0 / 1500


Batched teacher annotation: 100%|██████████| 1500/1500 [3:18:39<00:00,  7.95s/it]

Done.


# Validation: parse-failure rate + spot-check

In [23]:
def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            records.append(json.loads(line))
    return records

kd_dataset = load_jsonl(OUTPUT_PATH)
print(f"Total records: {len(kd_dataset)}")

n_failed = sum(1 for r in kd_dataset if r["parse_failed"])
print(f"Parse failures: {n_failed} ({n_failed/len(kd_dataset)*100:.1f}%)")

n_positive = sum(1 for r in kd_dataset if r["annotated_output_parsed"] and r["annotated_output_parsed"]["contains_IF"])
n_negative = len(kd_dataset) - n_failed - n_positive
print(f"contains_IF=True: {n_positive}, contains_IF=False: {n_negative}, failed: {n_failed}")

sent_counts = [
    len(r["annotated_output_parsed"].get("incidental_sentences", []))
    for r in kd_dataset if r["annotated_output_parsed"]
]
print(f"Avg incidental sentences per report: {sum(sent_counts)/len(sent_counts):.2f}")
print(f"Max: {max(sent_counts)}, reports with 0: {sum(1 for c in sent_counts if c == 0)}")

print("\n--- Spot check: 3 random parsed examples ---")
for r in random.sample([r for r in kd_dataset if r["annotated_output_parsed"]], 3):
    print(f"\n[{r['report_id']}]")
    print("Report snippet:", r["free_text"][:200], "...")
    print("Parsed output:", json.dumps(r["annotated_output_parsed"], indent=2))

print("\n--- Failed parses (if any) ---")
for r in [r for r in kd_dataset if r["parse_failed"]][:3]:
    print(f"\n[{r['report_id']}] raw output:", r["annotated_output_raw"][:300])

Total records: 1500
Parse failures: 2 (0.1%)
contains_IF=True: 256, contains_IF=False: 1242, failed: 2
Avg incidental sentences per report: 0.27
Max: 9, reports with 0: 1243

--- Spot check: 3 random parsed examples ---

[R1209]
Report snippet: The patient has a significantly enlarged spleen, measuring a volume of 676.4 cc, with a mean Hounsfield Unit (HU) value of 93.8 +/- 25.2. The liver is of normal size, with a volume of 2069.0 cc and a  ...
Parsed output: {
  "contains_IF": false,
  "incidental_sentences": []
}

[R0571]
Report snippet: The patient has an enlarged spleen, measuring 321.7 cc in volume, with a mean HU value of 106.4 +/- 32.8. The liver is of normal size, with a volume of 1675.9 cc and a mean HU value of 106.5 +/- 25.5. ...
Parsed output: {
  "contains_IF": false,
  "incidental_sentences": []
}

[R0014]
Report snippet: The patient's spleen is of normal size, measuring 266.6 cc in volume, with a mean Hounsfield Unit (HU) value of 71.7 +/- 26.7. The liver is also of nor

# Dataset construction

In [24]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# Verify teacher/student share vocab
assert proposer_tok.vocab_size == AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True).vocab_size, \
    "Tokenizer mismatch -- teacher and student vocabs differ, top-k alignment will be invalid."
print(f"Vocab size match confirmed: {proposer_tok.vocab_size}")

# Reversed pipeline: KD runs first on a PLAIN 0.5B base. There's no existing
# QLoRA adapter to load and continue training, so we attach a fresh LoRA
# adapter purely to make the 4-bit base trainable on a T4. This adapter gets
# merged into the base weights at the end of the notebook (see the "Merge KD
# weights into base" cell below) -- that merged checkpoint is what notebook 2
# treats as its starting "base model" for QLoRA.
KD_LORA_CONFIG = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

def load_estimator_for_training():
    tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    tok.pad_token = tok.eos_token
    tok.padding_side = "right"  # right-padding for training

    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID, quantization_config=bnb_config_t4, device_map={"": 0}, trust_remote_code=True,
    )
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    model = get_peft_model(base, KD_LORA_CONFIG)
    model.train()
    model.print_trainable_parameters()
    return model, tok

student_model, student_tok = load_estimator_for_training()
gpu_report("after student load (training mode, fresh LoRA)")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size match confirmed: 151643


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 4,325,376 || all params: 498,358,144 || trainable%: 0.8679
[after student load (training mode, fresh LoRA)] GPU 0: 6.34 GB / 15.64 GB
[after student load (training mode, fresh LoRA)] GPU 1: 0.00 GB / 15.64 GB


In [25]:
# Student now sees the SAME ACR-guideline context the teacher saw, injected into
# its system prompt at both train time (here) and inference time (eval cells
# below) -- mod #3. The earlier run likely underperformed partly because the
# student had no external knowledge source at inference and had to rely
# purely on whatever got baked into its weights during tuning; this closes
# that gap by giving it the same retrieved ACR context the 7B teacher used
# when it generated the soft labels the student is distilling from.
STUDENT_ACR_TOP_K = 10

def build_student_instructions(report_text, acr_index=ACR_INDEX, top_k=STUDENT_ACR_TOP_K):
    filtered_context = build_filtered_acr_context(report_text, acr_index, top_k=top_k)
    return TASK_INSTRUCTIONS_TEMPLATE.format(acr_context=filtered_context)

def build_student_prompt_ids(tokenizer, report_text, acr_index=ACR_INDEX):
    messages = [
        {"role": "system", "content": build_student_instructions(report_text, acr_index)},
        {"role": "user", "content": f"Report:\n{report_text}"},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    return ids

class KDDataset(torch.utils.data.Dataset):
    def __init__(self, kd_records, tokenizer, acr_index=ACR_INDEX, max_length=2048):
        self.tokenizer = tokenizer
        self.acr_index = acr_index
        self.max_length = max_length
        self.examples = [r for r in kd_records if not r["parse_failed"] and len(r["token_distributions"]) > 0]
        print(f"KDDataset: {len(self.examples)} usable examples (dropped {len(kd_records)-len(self.examples)} parse failures)")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        rec = self.examples[idx]
        prompt_ids = build_student_prompt_ids(self.tokenizer, rec["free_text"], self.acr_index)
        target_ids = [d["generated_token_id"] for d in rec["token_distributions"]]
        eos_id = self.tokenizer.eos_token_id
        if target_ids[-1] != eos_id:
            target_ids = target_ids + [eos_id]  # ensure student learns to stop

        input_ids = prompt_ids + target_ids
        labels = [-100] * len(prompt_ids) + target_ids

        if len(input_ids) > self.max_length:
            # truncate from the left of the prompt, never from the target
            overflow = len(input_ids) - self.max_length
            input_ids = input_ids[overflow:]
            labels = labels[overflow:]
            prompt_len_adj = len(prompt_ids) - overflow
        else:
            prompt_len_adj = len(prompt_ids)

        teacher_topk_ids = [d["topk_token_ids"] for d in rec["token_distributions"]]
        teacher_topk_probs = [d["topk_probs"] for d in rec["token_distributions"]]
        teacher_topk_mass = [d["topk_mass"] for d in rec["token_distributions"]]

        return {
            "input_ids": input_ids,
            "labels": labels,
            "target_start": prompt_len_adj,       # index in sequence where target span begins
            "target_len": len(target_ids) - (1 if target_ids[-1] == eos_id and rec["token_distributions"][-1]["generated_token_id"] != eos_id else 0),
            "teacher_topk_ids": teacher_topk_ids,
            "teacher_topk_probs": teacher_topk_probs,
            "teacher_topk_mass": teacher_topk_mass,
        }

def kd_collate_fn(batch, pad_token_id):
    max_len = max(len(ex["input_ids"]) for ex in batch)
    input_ids, attention_mask, labels = [], [], []
    for ex in batch:
        pad_n = max_len - len(ex["input_ids"])
        input_ids.append(ex["input_ids"] + [pad_token_id] * pad_n)
        attention_mask.append([1] * len(ex["input_ids"]) + [0] * pad_n)
        labels.append(ex["labels"] + [-100] * pad_n)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "meta": batch,  # keep raw per-example teacher info (variable length) for the KL loss step
    }

kd_train_dataset = KDDataset(kd_dataset, student_tok)

KDDataset: 1498 usable examples (dropped 2 parse failures)


# Combined loss: hard-label CE + bucketed top-k KL divergence

In [26]:
ALPHA = 0.3      # weight on hard-label vs KL: loss = ALPHA * focal + (1 - ALPHA) * KL  (tune later)
KD_TEMPERATURE = 1.0  # softening applied to both teacher and student before KL (classic Hinton KD; tune later)
FOCAL_GAMMA = 2.0     # standard focal-loss focusing parameter

def soften(probs, T):
    """Approximate temperature softening on already-normalized probabilities (probs^(1/T), renormalized).
    Note: this is an approximation since we stored post-softmax probs, not raw logits."""
    if T == 1.0:
        return probs
    softened = probs.pow(1.0 / T)
    return softened / softened.sum(dim=-1, keepdim=True)

def focal_cross_entropy(logits, targets, gamma=FOCAL_GAMMA, ignore_index=-100):
    """Token-level focal loss -- drop-in replacement for the hard-label F.cross_entropy
    term (mod #2). Down-weights tokens the model already gets confidently right (most of
    the fixed JSON skeleton) so gradient concentrates on the harder, minority-class tokens
    -- i.e. the actual incidental-finding content, underrepresented given the ~75/25 split."""
    log_probs = F.log_softmax(logits, dim=-1)
    ce = F.nll_loss(log_probs, targets, ignore_index=ignore_index, reduction="none")
    valid_mask = (targets != ignore_index).float()
    with torch.no_grad():
        pt = log_probs.gather(1, targets.clamp(min=0).unsqueeze(1)).squeeze(1).exp()
    focal_weight = (1 - pt).pow(gamma)
    loss = focal_weight * ce * valid_mask
    denom = valid_mask.sum().clamp(min=1)
    return loss.sum() / denom

def compute_kd_loss(model, batch, device):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits  # (B, L, V)

    # --- Hard-label focal loss (standard causal LM shift) ---
    shift_logits = logits[:, :-1, :]
    shift_labels = labels[:, 1:]
    hard_loss = focal_cross_entropy(
        shift_logits.reshape(-1, shift_logits.size(-1)),
        shift_labels.reshape(-1),
    )

    # --- Bucketed top-k KL, per example (variable target length, so loop) ---
    kl_losses = []
    for b, ex in enumerate(batch["meta"]):
        t_start = ex["target_start"]
        t_len = len(ex["teacher_topk_ids"])
        if t_len == 0:
            continue
        # position predicting target token j (0-indexed) is shift_logits index (t_start - 1 + j)
        start_idx = t_start - 1
        end_idx = start_idx + t_len
        if end_idx > shift_logits.size(1):
            end_idx = shift_logits.size(1)
            t_len = end_idx - start_idx
        if t_len <= 0:
            continue

        student_step_logits = shift_logits[b, start_idx:end_idx, :]  # (t_len, V)
        student_probs = F.softmax(student_step_logits.float(), dim=-1)

        topk_ids = torch.tensor(ex["teacher_topk_ids"][:t_len], device=device)      # (t_len, k)
        teacher_topk_probs = torch.tensor(ex["teacher_topk_probs"][:t_len], device=device, dtype=torch.float32)  # (t_len, k)
        teacher_mass = torch.tensor(ex["teacher_topk_mass"][:t_len], device=device, dtype=torch.float32)  # (t_len,)

        student_topk_probs = torch.gather(student_probs, 1, topk_ids)  # (t_len, k)
        student_other = (1.0 - student_topk_probs.sum(dim=-1, keepdim=True)).clamp(min=1e-8)
        teacher_other = (1.0 - teacher_mass).clamp(min=1e-8).unsqueeze(-1)

        # build (k+1)-dim distributions: top-k bins + one "other" bin
        student_dist = torch.cat([student_topk_probs, student_other], dim=-1)
        teacher_dist = torch.cat([teacher_topk_probs, teacher_other], dim=-1)

        student_dist = soften(student_dist.clamp(min=1e-8), KD_TEMPERATURE)
        teacher_dist = soften(teacher_dist.clamp(min=1e-8), KD_TEMPERATURE)

        student_log_dist = student_dist.log()
        kl = F.kl_div(student_log_dist, teacher_dist, reduction="batchmean")
        kl_losses.append(kl)

    kl_loss = torch.stack(kl_losses).mean() if kl_losses else torch.tensor(0.0, device=device)
    total_loss = ALPHA * hard_loss + (1 - ALPHA) * kl_loss
    return total_loss, hard_loss.item(), kl_loss.item()

# Training Loop

In [27]:
from functools import partial
from torch.utils.data import DataLoader

BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 2e-4
NUM_EPOCHS = 2
SAVE_DIR = "/kaggle/working/kd_lora_adapter"
os.makedirs(SAVE_DIR, exist_ok=True)

train_loader = DataLoader(
    kd_train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=partial(kd_collate_fn, pad_token_id=student_tok.pad_token_id),
)

trainable_params = [p for p in student_model.parameters() if p.requires_grad]
print(f"Trainable params: {sum(p.numel() for p in trainable_params):,}")
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE)

device = next(student_model.parameters()).device
student_model.train()

global_step = 0
for epoch in range(NUM_EPOCHS):
    running_hard, running_kl, running_total = 0.0, 0.0, 0.0
    optimizer.zero_grad()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for step, batch in enumerate(pbar):
        loss, hard_val, kl_val = compute_kd_loss(student_model, batch, device)
        (loss / GRAD_ACCUM_STEPS).backward()

        running_hard += hard_val
        running_kl += kl_val
        running_total += loss.item()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1
            pbar.set_postfix(focal=f"{running_hard/(step+1):.4f}", kl=f"{running_kl/(step+1):.4f}",
                              total=f"{running_total/(step+1):.4f}")

    student_model.save_pretrained(f"{SAVE_DIR}/epoch_{epoch+1}")
    student_tok.save_pretrained(f"{SAVE_DIR}/epoch_{epoch+1}")
    print(f"Saved adapter checkpoint: {SAVE_DIR}/epoch_{epoch+1}")
    flush_gpu()

Trainable params: 4,325,376


Epoch 1/2:   0%|          | 0/1498 [00:00<?, ?it/s]`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Epoch 1/2: 100%|██████████| 1498/1498 [51:33<00:00,  2.06s/it, focal=0.0111, kl=0.0148, total=0.0137]


Saved adapter checkpoint: /kaggle/working/kd_lora_adapter/epoch_1


Epoch 2/2: 100%|██████████| 1498/1498 [51:34<00:00,  2.07s/it, focal=0.0048, kl=0.0065, total=0.0060]


Saved adapter checkpoint: /kaggle/working/kd_lora_adapter/epoch_2


# Merge KD weights into base model (output for notebook 2)

In [28]:
# This is the artifact notebook 2 consumes: a full, non-quantized 0.5B
# checkpoint that already carries the KD signal, saved to /kaggle/working so
# it can be packaged as a Kaggle dataset (kd-best-abdomen) and loaded as the
# "base model" for the QLoRA stage.
FINAL_KD_CHECKPOINT = f"{SAVE_DIR}/epoch_{NUM_EPOCHS}"   # swap to a different epoch if the fuzzy eval below prefers it
MERGED_OUT_DIR = "/kaggle/working/kd_merged_abdomen"

flush_gpu()
merge_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, torch_dtype=torch.float16, device_map={"": 0}, trust_remote_code=True,
)
merge_peft = PeftModel.from_pretrained(merge_base, FINAL_KD_CHECKPOINT)
merged_model = merge_peft.merge_and_unload()

os.makedirs(MERGED_OUT_DIR, exist_ok=True)
merged_model.save_pretrained(MERGED_OUT_DIR, safe_serialization=True)
student_tok.save_pretrained(MERGED_OUT_DIR)
print(f"Merged KD model saved to {MERGED_OUT_DIR}")
print("Download this folder (zip it) and upload as a Kaggle dataset at mythreyeeh/kd-best-abdomen "
      "for notebook 2 to consume.")

del merge_base, merge_peft, merged_model
flush_gpu()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged KD model saved to /kaggle/working/kd_merged_abdomen
Download this folder (zip it) and upload as a Kaggle dataset at mythreyeeh/kd-best-abdomen for notebook 2 to consume.


# Metric Functions

In [29]:
def metric(example, prediction, trace=None):
    try:
        pred = parse_output(prediction.result)
        gold_sentences = json.loads(example.result)["incidental_sentences"]
    except Exception:
        return 0.0
    if pred is None:
        return 0.0
    gold_set = set(s.strip().lower() for s in gold_sentences)
    pred_set = set(s.strip().lower() for s in (pred.get("incidental_sentences") or []))
    if not gold_set and not pred_set:
        return 1.0
    if not gold_set or not pred_set:
        return 0.0
    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

def metric_with_cleanup(example, pred, trace=None):
    result = metric(example, pred, trace)
    flush_gpu()
    return result

In [30]:
from difflib import SequenceMatcher

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def fuzzy_match_sets(gold_set, pred_set, threshold=0.85):
    """Greedy one-to-one fuzzy matching between gold and predicted sentences.
    Returns tp, fp, fn counts."""
    gold_list = list(gold_set)
    pred_list = list(pred_set)
    matched_gold = set()
    matched_pred = set()
    pairs = []
    for gi, g in enumerate(gold_list):
        for pi, p in enumerate(pred_list):
            sim = similarity(g, p)
            if sim >= threshold:
                pairs.append((sim, gi, pi))
    pairs.sort(key=lambda x: -x[0])
    for sim, gi, pi in pairs:
        if gi in matched_gold or pi in matched_pred:
            continue
        matched_gold.add(gi)
        matched_pred.add(pi)
    tp = len(matched_gold)
    fp = len(pred_list) - len(matched_pred)
    fn = len(gold_list) - len(matched_gold)
    return tp, fp, fn

def run_evaluation_fuzzy(results, label="", threshold=0.85):
    parse_failures = sum(r["parse_failed"] for r in results)
    valid = [r for r in results if not r["parse_failed"]]
    total_tp = total_fp = total_fn = 0
    neg_scores, pos_scores = [], []
    for r in valid:
        gold_set = set(s.strip().lower() for s in r["gold_sentences"])
        pred_set = set(s.strip().lower() for s in (r["pred_sentences"] or []))
        if len(gold_set) == 0:
            neg_scores.append(1.0 if len(pred_set) == 0 else 0.0)
        else:
            tp, fp, fn = fuzzy_match_sets(gold_set, pred_set, threshold=threshold)
            total_tp += tp; total_fp += fp; total_fn += fn
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
            pos_scores.append(f1)
    avg_neg = sum(neg_scores) / len(neg_scores) if neg_scores else 0.0
    avg_pos = sum(pos_scores) / len(pos_scores) if pos_scores else 0.0
    n_neg, n_pos = len(neg_scores), len(pos_scores)
    if n_neg > 0 and n_pos > 0:
        macro_f1 = (avg_neg + avg_pos) / 2
    elif n_neg > 0:
        macro_f1 = avg_neg
    elif n_pos > 0:
        macro_f1 = avg_pos
    else:
        macro_f1 = 0.0
    weighted_f1 = (n_neg * avg_neg + n_pos * avg_pos) / (n_neg + n_pos) if (n_neg + n_pos) > 0 else 0.0
    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    micro_f1 = (2 * micro_precision * micro_recall / (micro_precision + micro_recall)
                if (micro_precision + micro_recall) > 0 else 0.0)
    print(f"\n{'='*50}\n=== {label} (fuzzy threshold={threshold}) ===\n{'='*50}")
    print(f"Parse failures:      {parse_failures}/{len(results)}")
    print(f"Negative-report acc: {avg_neg:.4f}  (n={n_neg})")
    print(f"Positive-report F1:  {avg_pos:.4f}  (n={n_pos})")
    print(f"Sentence Macro F1:   {macro_f1:.4f}")
    print(f"Sentence Weighted:   {weighted_f1:.4f}")
    print(f"Sentence Micro F1:   {micro_f1:.4f}")
    return {"macro_f1": macro_f1, "weighted_f1": weighted_f1, "micro_f1": micro_f1,
            "parse_failures": parse_failures, "n_neg": n_neg, "n_pos": n_pos,
            "avg_neg": avg_neg, "avg_pos": avg_pos}

# Load trained student checkpoint for inference

In [31]:
EVAL_CHECKPOINT = FINAL_KD_CHECKPOINT  # adapter checkpoint (pre-merge); swap to a different epoch if desired

eval_tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
eval_tok.pad_token = eval_tok.eos_token
eval_tok.padding_side = "left"

eval_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, quantization_config=bnb_config_t4, device_map={"": 0}, trust_remote_code=True,
)
eval_model = PeftModel.from_pretrained(eval_base, EVAL_CHECKPOINT)
eval_model.eval()
gpu_report("after eval student load")

def generate_student_prediction(model, tokenizer, report_text, acr_index=ACR_INDEX, max_new_tokens=768):
    messages = [
        {"role": "system", "content": build_student_instructions(report_text, acr_index)},
        {"role": "user", "content": f"Report:\n{report_text}"},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
        )
    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[after eval student load] GPU 0: 6.90 GB / 15.64 GB
[after eval student load] GPU 1: 0.00 GB / 15.64 GB


# Run inference over test records

In [32]:
student_eval_results = []
for rec in tqdm(test_dataset, desc="Student eval inference"):
    raw_output = generate_student_prediction(eval_model, eval_tok, rec["free_text"])
    parsed = parse_output(raw_output)
    student_eval_results.append({
        "report_id": rec["report_id"],
        "gold_sentences": rec["gold"].get("incidental_sentences", []),
        "pred_sentences": parsed.get("incidental_sentences") if parsed else None,
        "parse_failed": parsed is None,
        "raw_output": raw_output,
    })

run_evaluation_fuzzy(student_eval_results, label="KD student (fuzzy)", threshold=0.85)

Student eval inference: 100%|██████████| 100/100 [02:26<00:00,  1.47s/it]


=== KD student (fuzzy) (fuzzy threshold=0.85) ===
Parse failures:      0/100
Negative-report acc: 1.0000  (n=61)
Positive-report F1:  0.0000  (n=39)
Sentence Macro F1:   0.5000
Sentence Weighted:   0.6100
Sentence Micro F1:   0.0000


{'macro_f1': 0.5,
 'weighted_f1': 0.61,
 'micro_f1': 0.0,
 'parse_failures': 0,
 'n_neg': 61,
 'n_pos': 39,
 'avg_neg': 1.0,
 'avg_pos': 0.0}